# Deploy llm-d Intelligent Routing

This notebook deploys **Qwen3-Coder-30B** with the **llm-d Endpoint Picker (EPP)** on OpenShift AI. After completion, inference requests will be routed based on prefix-cache affinity, KV-cache utilization, and queue depth — rather than simple round-robin.

**What we'll do:**
1. Verify prerequisites (RHOAI 3.3+, Service Mesh, LeaderWorkerSet, cert-manager)
2. Create namespace, HF token secret, and model cache PVC
3. Deploy the LLMInferenceService with EPP scoring configuration
4. Verify pods, services, InferencePool, and HTTPRoute
5. Test inference through the llm-d gateway
6. Verify prefix-cache behavior with repeated prompts

## 1. Verify Prerequisites

llm-d requires several operators and CRDs beyond the basic RHOAI installation. Let's confirm they're present.

In [ ]:
%%bash
echo "=== RHOAI Operator ==="
oc get csv -n redhat-ods-operator 2>/dev/null | grep -E 'rhods|openshift-ai' || echo "⚠️  RHOAI not found"

echo ""
echo "=== Service Mesh ==="
oc get servicemeshcontrolplane -A 2>/dev/null | head -5 || echo "⚠️  Service Mesh not found"

echo ""
echo "=== LeaderWorkerSet CRD ==="
oc get crd leaderworkersets.leaderworkerset.x-k8s.io 2>/dev/null && echo "✅ Present" || echo "⚠️  Missing — install LeaderWorkerSet Operator"

echo ""
echo "=== cert-manager CRD ==="
oc get crd certificates.cert-manager.io 2>/dev/null && echo "✅ Present" || echo "⚠️  Missing — install cert-manager"

echo ""
echo "=== LLMInferenceService CRD ==="
oc get crd llminferenceservices.serving.kserve.io 2>/dev/null && echo "✅ Present" || echo "⚠️  Missing — requires RHOAI 3.3+"

## 2. Create Namespace, Secret, and Model Cache PVC

The llm-d model deployment needs:
- A dedicated namespace (`llm-d-serving`)
- A HuggingFace token secret for model download
- A persistent volume for model weights (avoids re-download on pod restart)

In [ ]:
%%bash
oc apply -f manifests/00-namespace-pvc.yaml

echo ""
echo "=== Namespace ==="
oc get namespace llm-d-serving

echo ""
echo "=== Secret ==="
oc get secret hf-token -n llm-d-serving

echo ""
echo "=== PVC (100Gi for model weights) ==="
oc get pvc model-cache -n llm-d-serving

## 3. Deploy LLMInferenceService

The `LLMInferenceService` CRD provisions the full llm-d stack in one resource:
- vLLM model server pods with prefix caching and tool calling enabled
- EPP scheduler with scoring weights (prefix-cache ×3, KV-cache ×2, queue ×2)
- InferencePool for endpoint registration
- HTTPRoute through the MaaS gateway

> **Note:** Model download takes 4–8 minutes on first deploy. The PVC caches weights for instant restarts afterward.

In [ ]:
%%bash
oc apply -f manifests/01-llminferenceservice.yaml

echo ""
echo "Waiting for model pod to start (this takes 4-8 min on first deploy)..."
echo "Monitor with: oc get pods -n llm-d-serving -w"

## 4. Verify Deployment

Check that all components are running: vLLM pod(s), EPP scheduler, InferencePool, and HTTPRoute.

In [ ]:
%%bash
echo "=== Pods ==="
oc get pods -n llm-d-serving

echo ""
echo "=== Services ==="
oc get svc -n llm-d-serving

echo ""
echo "=== InferencePool ==="
oc get inferencepool -n llm-d-serving

echo ""
echo "=== HTTPRoute ==="
oc get httproute -n llm-d-serving

echo ""
echo "=== LLMInferenceService Status ==="
oc get llminferenceservice -n llm-d-serving

## 5. Test Inference

Send a test chat completion request through the MaaS gateway to verify the end-to-end path: MaaS → llm-d EPP → vLLM.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(oc get ingresses.config cluster -o jsonpath='{.spec.domain}')
MODEL_URL=$(oc get llminferenceservice qwen3-coder-fp8 -n llm-d-serving -o jsonpath='{.status.url}' 2>/dev/null)

if [ -z "$MODEL_URL" ]; then
  MODEL_URL="https://maas.${CLUSTER_DOMAIN}"
  echo "Using MaaS gateway: $MODEL_URL"
fi

echo "Sending test request..."
curl -sk "${MODEL_URL}/v1/chat/completions" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8",
    "messages": [{"role": "user", "content": "Write a Python hello world function."}],
    "max_tokens": 100
  }' | python3 -m json.tool

## 6. Verify Prefix-Cache Behavior

Prefix caching is the key advantage of llm-d for coding assistants. When multiple requests share the same system prompt, EPP routes them to pods that already hold that prefix in KV cache — reducing TTFT significantly.

Let's send the same system prompt multiple times and check the prefix cache metrics.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(oc get ingresses.config cluster -o jsonpath='{.spec.domain}')
MODEL_URL=$(oc get llminferenceservice qwen3-coder-fp8 -n llm-d-serving -o jsonpath='{.status.url}' 2>/dev/null)
[ -z "$MODEL_URL" ] && MODEL_URL="https://maas.${CLUSTER_DOMAIN}"

SYSTEM_PROMPT="You are a senior Python developer. Follow PEP 8, use type hints, write docstrings for all public functions."

echo "Sending 5 requests with identical system prompt..."
for i in 1 2 3 4 5; do
  echo "  Request $i..."
  curl -sk "${MODEL_URL}/v1/chat/completions" \
    -H "Content-Type: application/json" \
    -d "{
      \"model\": \"Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8\",
      \"messages\": [
        {\"role\": \"system\", \"content\": \"${SYSTEM_PROMPT}\"},
        {\"role\": \"user\", \"content\": \"Write a function to check if request $i is a palindrome.\"}
      ],
      \"max_tokens\": 50
    }" -o /dev/null -w "  TTFT: %{time_starttransfer}s  Total: %{time_total}s\n"
done

echo ""
echo "=== Prefix Cache Metrics (from vLLM pod) ==="
VLLM_POD=$(oc get pods -n llm-d-serving -l app.kubernetes.io/name=qwen3-coder-fp8 -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
if [ -n "$VLLM_POD" ]; then
  oc exec -n llm-d-serving $VLLM_POD -- curl -s localhost:8000/metrics 2>/dev/null | grep -E "prefix_cache" || echo "(Metrics may require HTTPS port)"
fi

## Summary

| Component | Status |
|-----------|--------|
| Namespace `llm-d-serving` | Created |
| Model cache PVC (100Gi) | Bound |
| LLMInferenceService | Deployed |
| EPP Scheduler | Running |
| InferencePool | Active |
| Prefix caching | Enabled |
| Tool calling (`qwen3_coder`) | Enabled |

**Key observations from prefix cache test:**
- First request has higher TTFT (cold prefix)
- Subsequent requests with the same system prompt should show lower TTFT (prefix cache hit)
- EPP routes same-prefix requests to the same pod

→ Continue to `3_verify_routing.ipynb` to analyze routing decisions, concurrent load distribution, and prefix-cache effectiveness in detail.